# 교안 02: 계획을 세우고 실행하기 (MCP 도구로)

앞 시간에 우리는 **남이 만든 MCP 서버**를 하나씩 붙여, 파일·DB·코드 실행·브라우저를 에이전트에 쥐여 줬습니다.
그런데 그때 던진 질문은 대개 **한두 단계**였습니다. 실무의 분석 요청은 다릅니다.
"조회하고, 계산하고, 그래프까지 그려서 리포트로 정리해줘". 이런 **복합 요청**을 에이전트가 우왕좌왕하지 않게 하려면,
**먼저 계획을 세우고** 단계별로 처리하게 해야 합니다.

이번 시간엔 에이전트에 **미들웨어(middleware)** 를 끼워 **할 일 목록(계획)** 을 세우게 하고,
**수집 → 분석·통계 → 시각화 → 리포트** 로 이어지는 자동 분석 파이프라인을 MCP 도구만으로 완성합니다.

## 지난 시간 복습

- **MCP 서버**를 붙이면 우리가 만들지 않은 도구를 그대로 씁니다. 설정은 `command`·`args`·`transport` 딕셔너리 하나입니다.
- 노트북에서는 `await client.get_tools()` 로 도구를 받습니다(상태 있는 서버만 세션을 열어 둡니다).
- 에이전트에 **넘긴 도구만** 쓸 수 있습니다. 도구를 고르는 것이 곧 권한 설계였습니다.
- 이번 시간엔 그 도구들을 **그대로 쓰되**, 에이전트가 **계획을 세우게** 만듭니다.

## 오늘의 목표

- [ ] **복합 요청**을 계획 없이 처리할 때의 한계를 관찰한다.
- [ ] **미들웨어**가 무엇인지, 에이전트의 어디를 확장하는지 이해한다.
- [ ] **`TodoListMiddleware`** 로 **Plan-and-Execute**(계획 수립 → 단계 실행)를 구현하고, 그 계획을 눈으로 확인한다.
- [ ] **수집 → 분석·통계 → 시각화 → 리포트** end-to-end 자동 분석을 MCP 도구로 실행한다.
- [ ] 분석 결과를 **정해진 틀**로 바꿔 표·JSON 으로 적재한다.

---
## 준비

이 단원은 **MCP 서버 세 개**를 씁니다. 앞 단원에서 하나씩 다뤄 본 것들입니다.

| 서버 | 실행 | 맡는 일 | 공식 문서 |
|---|---|---|---|
| SQLite `mcp-server-sqlite` | `uvx` | 조회·집계 | https://pypi.org/project/mcp-server-sqlite/ |
| 코드 실행 `mcp-server-code-runner` | `npx` | 계산·그래프 | https://github.com/formulahendry/mcp-server-code-runner |
| 파일시스템 `@modelcontextprotocol/server-filesystem` | `npx` | 리포트 저장 | https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem |

준비물은 **Node.js**(`npx`)·**uv**(`uvx`)·**`OPENAI_API_KEY`**(일차 폴더의 `.env`)입니다.

In [14]:
import platform
import sys
from pathlib import Path

DAY_DIR = Path.cwd().parent        # 이 노트북은 교안_02 폴더에서 연다. 그 위가 일차 폴더(day21).
sys.path.append(str(DAY_DIR))      # 일차 폴더의 utils.py 를 쓴다

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, TodoListMiddleware
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from utils import (child_env, chinook_db_path, load_api_key,
                   print_trajectory, quiet_stdio_logs, tool_names)

quiet_stdio_logs()                 # 코드 실행 서버가 stdout 에 섞어 보내는 안내문 때문에 나는 긴 경고를 끈다

DATA_DIR = DAY_DIR / "data"

load_api_key(DAY_DIR)              # 모델을 부르므로 키를 맨 앞에서 확인한다
DB_PATH = chinook_db_path(DATA_DIR)   # data 폴더의 chinook.db 경로를 돌려준다
CHILD_ENV = child_env()                 # 코드 실행 서버가 python 을 찾게 하는 환경 변수

chinook DB 준비됨: chinook.db (864KB)


In [11]:
# 앞 단원에서 한 줄씩 뜯어본 설정에 한 가지를 더한다. 두 서버를 output 폴더 기준으로 띄우는 것이다.
# 산출물은 모두 일차 폴더의 output 에 모은다.
#   files_write_file : 파일 서버에 열어 준 폴더가 상대경로의 기준이 된다
#   code_run-code    : 서버 프로세스의 작업 폴더(cwd)가 상대경로의 기준이 된다
# 기준이 서로 다른 두 도구를 같은 폴더로 맞춰 두면, 모델은 파일 이름만 적으면 된다.
# 긴 경로를 프롬프트에 넣지 않는 것이 중요하다. 넣으면 모델이 그 경로를 다시 타이핑하다 오타를 내고,
# 코드 실행 서버는 그 오타를 걸러 주지 않아 엉뚱한 폴더에 조용히 저장된다.
OUTPUT_DIR = DAY_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)      # 파일 서버는 없는 폴더를 만들어 주지 않는다
REPORT_NAME = "음악판매_리포트.md"
CHART_NAME = "연도별_매출.png"

SQLITE = {
    "command": "uvx",
    "args": ["--with", "mcp==1.9.4",         # 이 서버는 최신 mcp 로 띄우면 죽는다. 버전을 고정한다
             "--from", "mcp-server-sqlite",
             "mcp-server-sqlite",
             "--db-path", str(DB_PATH)],
    "transport": "stdio",
}
CODE_RUNNER = {
    "command": "npx",
    "args": ["-y", "mcp-server-code-runner"],
    "transport": "stdio",
    "env": CHILD_ENV,
    "cwd": str(OUTPUT_DIR),   # 이 서버가 돌리는 코드의 작업 폴더. savefig("그림.png") 가 여기에 떨어진다
}
FILESYSTEM = {
    "command": "npx",
    # 열어 주는 폴더가 곧 쓰기 허용 범위이자 상대경로의 기준이다
    "args": ["-y", "@modelcontextprotocol/server-filesystem", str(OUTPUT_DIR)],
    "transport": "stdio",
}


# 한글 폰트 이름은 OS 마다 다르다. "한글 폰트를 써라" 라고만 하면 모델이 없는 이름을 골라
# 제목이 네모(□□□)로 나온다. 여기서 정해 프롬프트에 넣어 준다.
FONT = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}.get(platform.system(), "NanumGothic")

In [12]:
print("서버 세 개를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
client = MultiServerMCPClient(
    {"db": SQLITE, "code": CODE_RUNNER, "files": FILESYSTEM},
    tool_name_prefix=True,      # db_·code_·files_ 접두사를 붙인다. 서버끼리 도구 이름이 겹쳐도 충돌하지 않는다
)
mcp_tools = await client.get_tools()

# 쓰기 도구는 리포트 저장용 하나만 남긴다. 넘기지 않은 도구는 모델이 존재조차 모른다.
ALLOWED = {"db_read_query", "db_list_tables", "db_describe_table",
           "code_run-code", "files_write_file"}
TOOLS = [t for t in mcp_tools if t.name in ALLOWED]

print("에이전트에 쓸 도구:", [t.name for t in TOOLS])

서버 세 개를 띄우는 중입니다(첫 실행은 오래 걸립니다)...
에이전트에 쓸 도구: ['db_read_query', 'db_list_tables', 'db_describe_table', 'code_run-code', 'files_write_file']


> 실습 데이터는 앞 단원에서 쓴 **chinook**(가상의 음악 판매점) DB 입니다.
> `invoices`(주문)·`customers`(고객)·`employees`(직원)·`tracks`(곡) 같은 표 11개가 이어져 있습니다.

---
# 1. 계획 없는 에이전트의 한계

여러 단계가 필요한 요청을 계획 없이 던지면, 에이전트가 **어디까지 했는지 놓치거나** 일부 단계를 **건너뛰기** 쉽습니다.
사람도 복잡한 일을 할 때 **할 일 목록**을 먼저 적는 이유와 같습니다.

먼저 **계획 장치가 없는** 에이전트에 복합 요청을 던져, 어떻게 처리하는지 메시지 기록으로 봅니다.

In [13]:
SYSTEM_BASE = (
    "너는 데이터 분석 비서다. 표의 조회·집계는 db_read_query 로 SQL 을 실행해 구하고, "
    "그 결과를 가공하는 계산과 그래프는 code_run-code 로 한다. 숫자를 암산하거나 지어내지 않는다. "
    "코드의 마지막 줄은 반드시 print 로 출력한다. 값만 적은 줄은 아무것도 돌려주지 않는다. "
    "결과가 비어 있으면 print 를 빠뜨린 것이니 print 를 넣어 다시 실행한다. "
    "경고(Stderr)만 돌아오면 이 서버가 표준 출력을 버린 것이다. 코드 맨 위에서 "
    "warnings.filterwarnings('ignore') 로 경고를 끄고 다시 실행한다. "
    "같은 코드를 두 번 보내지 않는다. "
    "code_run-code 는 한 번에 하나씩만 부른다. 이 서버는 모든 코드를 같은 임시 파일에 쓰므로 "
    "동시에 두 번 부르면 서로의 코드를 덮어써 실패한다. "
    "표나 열 이름이 확실하지 않으면 db_list_tables 와 db_describe_table 로 먼저 확인한다. "
    "그림은 files_write_file 로 만들 수 없다. 그림은 code_run-code 안에서 matplotlib 의 "
    "savefig 로 저장하고, 저장한 뒤 os.path.getsize 로 크기를 print 해 0 이 아닌지 확인한다. "
    "그래프 코드는 맨 위에 다음 두 줄을 그대로 넣는다. 창을 띄우지 않고, 한글이 네모로 깨지지 않게 하려는 것이다.\n"
    "matplotlib.use('Agg')\n"
    f"plt.rcParams['font.family'] = '{FONT}'\n"
    "마크다운·텍스트만 files_write_file 로 저장한다. "
    "파일을 저장할 때는 폴더 경로를 적지 않는다. 파일 이름만 적는다. "
    "두 도구 모두 저장 폴더에서 실행되므로 이름만 적으면 그 폴더에 저장된다."
)

# 계획 장치(middleware) 없이 도구만 붙인 에이전트 - 비교용
# timeout 을 준다. 기본값은 요청 하나를 10분까지 기다리고 두 번 더 재시도해서,
# 응답이 늦거나 분당 한도에 걸리면 화면만 보고는 멈춘 것과 구별되지 않는다.
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=60)

# 모델 호출 횟수 상한. 반복에 빠져도 예외 없이 상한에서 스스로 끝난다.
LIMIT = ModelCallLimitMiddleware(run_limit=20, exit_behavior="end")
plain_agent = create_agent(model, TOOLS, system_prompt=SYSTEM_BASE, middleware=[LIMIT])
print("계획 없는 에이전트 준비 완료")

계획 없는 에이전트 준비 완료


In [ ]:
# 한 문장에 네 가지 일(집계 2개·계산·그래프)을 담은 복합 요청
complex_q = ("chinook DB 를 분석해줘. 연도별 매출 합계와 국가별 매출 상위 5개를 구하고, "
             "연도별 매출의 전년 대비 증감률도 계산하고, 연도별 매출 막대그래프도 저장해줘.")

result_plain = await plain_agent.ainvoke({"messages": complex_q})
print_trajectory(result_plain)
print("\n불린 도구:", tool_names(result_plain))

> 계획 장치가 없으면 결과에 **`todos`(할 일 목록)가 없습니다**. 에이전트가 도구를 이어 부르긴 하지만,
> **무엇을 언제 할지 계획을 명시적으로 남기지 않습니다**. 단계가 더 많아지면 놓치는 일이 생깁니다.
> 실제로 위 기록에서 **그래프 저장이 빠졌는지** 확인해 보세요. 이제 **계획을 세우는 장치**를 끼워 봅니다.

---
# 2. 에이전트의 동작을 확장하는 미들웨어

## 미들웨어는 무엇인가

`create_agent` 가 만드는 에이전트는 **모델 호출 → 도구 실행 → 다시 모델 호출** 을 반복하는 루프입니다.
**미들웨어(middleware)** 는 이 루프의 코드를 고치지 않고, 루프의 정해진 지점에서 **내 코드가 대신 실행되게** 하는 장치입니다.

쓰는 법은 두 단계입니다.

1. `AgentMiddleware` 를 **상속**한 클래스를 만들고, **정해진 이름의 메서드**를 구현합니다.
2. 그 인스턴스를 `create_agent(..., middleware=[...])` 에 넘깁니다.

메서드 이름이 정해져 있는 이유는, 에이전트가 실행 중에 **그 이름으로 메서드를 찾아 호출**하기 때문입니다.
구현하지 않은 메서드는 기본 구현(아무것도 하지 않음)이 쓰이므로, **필요한 지점만 골라 구현**합니다.

## 훅: 미들웨어가 걸리는 자리

라이브러리가 루프에 **미리 열어 둔 자리**를 **훅(hook)** 이라고 합니다. 이름이 정해진 메서드 하나가 훅 하나입니다.

**훅은 자리, 미들웨어는 그 자리에 거는 물건**입니다. 훅은 라이브러리가 정해 두고, 우리는 미들웨어를 만들어 **필요한 훅에만** 겁니다.
미들웨어 하나가 훅을 **여러 개** 쓸 수도, 한 훅에 미들웨어가 **여러 개** 걸릴 수도 있습니다.

그래서 미들웨어를 만들 때 첫 질문은 "무엇을 할까" 가 아니라 **"어느 훅에서 해야 할 수 있는 일인가"** 입니다.
자리를 잘못 고르면 그 자리에서는 그 일을 할 방법이 아예 없습니다. 훅은 다음 **여섯 개**가 전부입니다.

이 방식을 쓰는 이유는 둘입니다. 에이전트 루프는 라이브러리 코드라 **직접 고치면 버전이 오를 때 깨지고**,
여러 기능을 각자 고쳐 넣으면 **서로 충돌**합니다. 확장 지점을 규격으로 정해 두면 두 문제가 모두 없습니다.

> MCP 와 같은 해법입니다. MCP 는 **도구를 붙이는 규격**을, 미들웨어는 **루프에 끼어드는 규격**을 정해 두었습니다.

## 구현할 수 있는 여섯 훅

<img src="../images/middleware_hooks.png" width="860">

| 훅(메서드) | 언제 호출되나 | 인자 → 반환값 |
|---|---|---|
| `before_agent` | 실행 시작 전 **1번** | `(state, runtime)` → 상태 변경분 `dict` 또는 `None` |
| `before_model` | 모델을 부르기 전 **매번** | `(state, runtime)` → 상태 변경분 `dict` 또는 `None` |
| `wrap_model_call` | 모델 호출을 **감싸서** | `(request, handler)` → 모델 응답 |
| `after_model` | 모델이 답한 뒤 **매번** | `(state, runtime)` → 상태 변경분 `dict` 또는 `None` |
| `wrap_tool_call` | 도구 호출을 **감싸서** | `(request, handler)` → 도구 결과 |
| `after_agent` | 실행이 끝난 뒤 **1번** | `(state, runtime)` → 상태 변경분 `dict` 또는 `None` |

여섯 개를 다 구현하지 않습니다. 대개 한두 개만 구현합니다.

## `before_`/`after_` 와 `wrap_` 은 할 수 있는 일이 다르다

인자를 보면 차이가 드러납니다.

- **`before_`/`after_`** 는 `state` 를 받습니다. 그 시점의 **상태를 읽고 바꾸는 것**까지 할 수 있습니다.
  호출 자체에는 손대지 못합니다.
- **`wrap_`** 은 `handler`, 곧 **호출 그 자체**를 인자로 받습니다. 그래서 `handler` 를 **여러 번 부르거나(재시도)
  한 번도 부르지 않을(차단)** 수 있고, 부르기 전에 요청을, 부른 뒤에 응답을 바꿀 수 있습니다.

재시도·대체 모델처럼 **호출을 다시 하거나 막아야 하는 기능**이 `wrap_` 자리에 있는 것은 이 때문입니다.

## 지점 말고도 붙는 것 두 가지

미들웨어는 실행 지점에 끼어드는 것 외에 두 가지를 더 할 수 있습니다.

- **도구를 추가한다**: `TodoListMiddleware` 는 `write_todos` 도구를 에이전트의 도구 목록에 넣습니다.
- **상태를 넓힌다**: 같은 미들웨어가 상태에 `todos` 키를 더해, 결과에서 `result["todos"]` 로 꺼낼 수 있게 합니다.

그래서 미들웨어 한 줄을 끼우면 동작만이 아니라 **모델이 쓸 수 있는 도구와 결과에 담기는 값**까지 함께 바뀝니다.

## `before`·`after` 와 `wrap` 의 차이

- **`before_model`** 은 **모델을 부르기 전에 할 일**, **`after_model`** 은 **모델이 답한 뒤에 할 일**을 적는 자리입니다. 모델 호출 자체는 라이브러리가 합니다.
- **`wrap_model_call`** 은 **모델 호출을 내가 하는** 자리입니다. 그래서 **안 부를 수도, 두 번 부를 수도** 있습니다.

```python
def before_model(self, state, runtime):
    ...                            # 부르기 전에 할 일만 적는다

def wrap_model_call(self, request, handler):
    ...                            # 부르기 전
    response = handler(request)    # <- 호출이 내 손에 있다
    ...                            # 부른 뒤
    return response
```

> 그래서 **재시도·대체 모델은 `wrap_` 에서만** 만들 수 있습니다. 한 번 더 부르려면 호출을 쥐고 있어야 하니까요.

## 여러 개를 끼우면 순서는

`middleware=[A(), B()]` 로 넘깁니다. **먼저 적은 A 가 바깥, B 가 안쪽**인 양파 구조입니다.

- `before_` 는 **적은 순서**(A → B)
- `after_` 는 **거꾸로**(B → A)
- `wrap_` 은 **A 가 B 를 감싼다**

여섯 훅을 모두 구현한 `A`·`B` 를 끼우고 도구를 한 번 쓰는 질문을 던지면 이렇게 찍힙니다.

```text
A.before_agent              <- 루프 밖, 한 번
B.before_agent
   A.before_model           <- 1번째 모델 호출
   B.before_model
   A.wrap_model_call  시작
   B.wrap_model_call  시작
   B.wrap_model_call  끝
   A.wrap_model_call  끝
   B.after_model            <- after 는 역순
   A.after_model
   A.wrap_tool_call   시작    <- 모델이 도구를 부르겠다고 해서 열렸다
   B.wrap_tool_call   시작
   B.wrap_tool_call   끝
   A.wrap_tool_call   끝
   A.before_model           <- 도구 결과를 들고 2번째 모델 호출
   ...(같은 순서로 반복)
B.after_agent               <- 루프 밖, 한 번
A.after_agent
```

## 어떤 미들웨어가 있나

| 갈래 | 미들웨어 | 하는 일 | 언제 |
|---|---|---|---|
| **계획** | `TodoListMiddleware` | 복합 요청에 **할 일 목록**을 세우고 추적 | **오늘** |
| **대화 관리** | `SummarizationMiddleware` | 길어진 대화의 **앞부분을 요약** | 52일차 |
| | `ContextEditingMiddleware` | 오래된 **도구 결과를 잘라** 냄 | 52일차 |
| **안전** | `HumanInTheLoopMiddleware` | 위험한 도구 앞에서 **사람의 승인** | 51일차 |
| | `PIIMiddleware` | 개인정보를 **가리거나 막음** | 52일차 |
| **한도** | `ModelCallLimitMiddleware` | 모델 **호출 횟수 상한** | **오늘** |
| | `ToolCallLimitMiddleware` | 도구 **호출 횟수 상한** | 참고 |
| **견고성** | `ModelRetryMiddleware` · `ToolRetryMiddleware` | 실패하면 **다시 시도** | 참고 |
| | `ModelFallbackMiddleware` | 실패하면 **다른 모델로** | 참고 |

> **고르는 기준**: 판단이 문제면 계획, 대화가 길면 대화 관리, 위험하면 안전, 폭주·요금이면 한도, 간헐적 실패면 견고성.
> 없으면 직접 만들 수도 있지만, 이 과정에서는 **골라 끼우는** 데까지 합니다.

> **한도 미들웨어는 이 강의에서 이미 쓰고 있습니다.** 1절과 앞에서 만든 에이전트에
> `ModelCallLimitMiddleware(run_limit=20, exit_behavior="end")` 를 끼웠습니다.
> 모델이 같은 도구 호출을 반복해도 20번에서 스스로 멈추고, `exit_behavior="end"` 라서
> 예외가 아니라 **그때까지의 기록이 그대로** 돌아옵니다. 상한이 없으면 반복이 끝나지 않아
> 화면이 멈춘 것처럼 보이고 요금만 쌓입니다.

> MCP 서버를 쓸 때 특히 쓸모 있는 것이 **`HumanInTheLoopMiddleware`** 입니다.
> 남의 서버가 준 도구에는 `write_query`·`write_file` 처럼 **바꾸는 도구**가 섞여 있습니다.
> 앞 단원에서는 그런 도구를 **아예 넘기지 않는** 방법을 썼고, 꼭 필요하면 이 미들웨어로 **사람의 승인**을 받게 합니다.

표가 맞는지 확인해 봅니다. 기본 클래스(`AgentMiddleware`)와 **달라진 훅**이 그 미들웨어가 쓰는 자리입니다.

In [4]:
from langchain.agents.middleware import (AgentMiddleware, HumanInTheLoopMiddleware,
                                         SummarizationMiddleware, TodoListMiddleware,
                                         ToolCallLimitMiddleware, ToolRetryMiddleware)

HOOKS = ['before_agent', 'before_model', 'wrap_model_call',
         'after_model', 'wrap_tool_call', 'after_agent']


def used_hooks(mw_class):
    """그 미들웨어가 실제로 구현한 훅 이름만 골라 돌려준다."""
    # 기본 클래스의 훅과 '다른 함수' 로 바뀌어 있으면 그 자리를 쓴 것이다
    return [h for h in HOOKS
            if getattr(mw_class, h) is not getattr(AgentMiddleware, h)]


for mw in (TodoListMiddleware, SummarizationMiddleware, HumanInTheLoopMiddleware,
           ToolCallLimitMiddleware, ToolRetryMiddleware):
    print(f'{mw.__name__:28} {used_hooks(mw)}')

TodoListMiddleware           ['wrap_model_call', 'after_model']
SummarizationMiddleware      ['before_model']
HumanInTheLoopMiddleware     ['after_model']
ToolCallLimitMiddleware      ['after_model']
ToolRetryMiddleware          ['wrap_tool_call']


하나같이 한두 자리만 씁니다. 그 자리가 왜 거기인지도 읽힙니다.

- **`SummarizationMiddleware`** → `before_model`: 대화를 줄이려면 **부르기 전**이어야 합니다.
- **`HumanInTheLoopMiddleware`** → `after_model`: 모델이 **도구를 부르겠다고 말한 직후**가 물어볼 자리입니다.
- **`ToolRetryMiddleware`** → `wrap_tool_call`: 다시 부르려면 호출을 쥐어야 합니다.
- **`TodoListMiddleware`**(오늘) → 두 자리를 씁니다. 아래에서 따로 봅니다.

### `TodoListMiddleware` 가 두 자리를 쓰는 이유

| 자리 | 하는 일 |
|---|---|
| `wrap_model_call` | 우리 시스템 프롬프트 **뒤에** "할 일 목록을 쓰라"는 안내문을 덧붙여 모델을 부른다 |
| `after_model` | 한 응답에 `write_todos` 가 **2개 이상**이면 오류로 되돌려 다시 답하게 한다 |

- **왜 `wrap_` 인가**: 요청을 바꿔서 불러야 하기 때문입니다. `before_model` 로는 호출 자체를 바꿀 수 없습니다.
- **왜 2개를 막나**: `write_todos` 는 목록을 **통째로 갈아 끼웁니다**. 한 응답에 두 번 들어오면 어느 쪽이 최종인지 정할 방법이 없습니다.

**계획을 세우게 만드는 일은 `wrap_model_call`, 계획이 한 번에 하나만 쓰이게 지키는 일은 `after_model`** 입니다.

> 미들웨어는 **도구를 함께 들고 오기도** 합니다. 계획을 적는 `write_todos` 가 바로 `TodoListMiddleware` 가 추가하는 도구입니다.
> 끼우는 것만으로 도구가 하나 늘어납니다. **MCP 서버가 도구를 주는 것과 같은 자리**에 도구가 하나 더 붙는 셈입니다.

### 🖐️ 함께 따라하기: 다른 미들웨어의 훅 자리 확인하기

데모는 다섯 개를 훑었습니다. 이번엔 **아직 안 본 미들웨어**의 자리를 직접 확인하고, 왜 거기인지 말로 적어 봅니다.

| 미들웨어 | 무엇을 해 주나 |
|---|---|
| `ModelCallLimitMiddleware` | 모델 호출 **횟수를 세어 상한을 넘기면 멈춘다**. 오늘 우리가 쓴 그 미들웨어입니다 |
| `ModelFallbackMiddleware` | 모델 호출이 **실패하면 다른 모델로 다시 부른다**. 넘긴 순서대로 넘어갑니다 |

1. `langchain.agents.middleware` 에서 두 미들웨어를 가져오세요.
2. 위에서 만든 `used_hooks()` 로 각각이 쓰는 훅을 출력하세요.
3. 출력된 자리를 보고, **왜 그 자리인지** 한 줄 주석으로 적어 보세요.

**확인 기준**: 하는 일이 다르니 자리도 다릅니다.

- `ModelCallLimitMiddleware` → `before_model`·`after_model`
  세는 일은 **부르기 전에 한도를 확인하고 부른 뒤에 하나 더하면** 됩니다. 호출을 쥘 필요가 없습니다.
- `ModelFallbackMiddleware` → `wrap_model_call`
  실패했을 때 **다른 모델로 다시 부르려면** 그 호출이 내 손에 있어야 합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) ModelCallLimitMiddleware 와 ModelFallbackMiddleware 를 가져온다
# 2) used_hooks 로 두 미들웨어가 쓰는 훅을 출력한다
# 3) 왜 그 자리인지 한 줄 주석으로 적는다

---
# 3. 계획을 세우는 `TodoListMiddleware`

## 이 방식의 이름은 Plan-and-Execute

**먼저 계획을 세우고(Plan), 그 계획을 한 단계씩 실행(Execute)** 하는 방식입니다.

지난 시간의 **ReAct** 와 견주면 차이가 하나입니다.

| | 전체 그림 | 단계가 많아지면 |
|---|---|---|
| **ReAct** | 그리지 않는다. "생각 → 행동 → 관찰" 을 한 걸음씩 반복할 뿐 | 무엇을 남겨 뒀는지 놓친다 |
| **Plan-and-Execute** | **할 일 목록을 먼저 문서로 남긴다** | 목록을 보며 움직이므로 진행 상황을 눈으로 추적할 수 있다 |

## 쓰는 법

LangChain 에서는 직접 구현하지 않고 **`TodoListMiddleware`** 를 끼워 얻습니다.
붙인 도구는 그대로 두고 두 가지만 바꿉니다.

1. `create_agent` 에 **`middleware=[TodoListMiddleware()]`** 를 넘긴다
2. 시스템 프롬프트에 **"복합 요청은 먼저 계획을 세워라"** 를 적는다

<img src="../images/react_vs_plan_execute.png" width="760">

*왼쪽은 지난 시간의 ReAct, 오른쪽이 오늘 만들 방식입니다. 차이는 **할 일 목록을 남기느냐** 하나입니다.*

In [7]:
# 도구는 앞에서 고른 그대로다. 달라지는 것은 middleware 한 줄과 프롬프트 한 문장이다.
SYSTEM_PLAN = SYSTEM_BASE + (
    " 여러 단계가 필요한 복합 요청은 먼저 계획을 세우고 단계별로 처리하라."
)

analyst = create_agent(model, TOOLS,
                       middleware=[TodoListMiddleware(), LIMIT],
                       system_prompt=SYSTEM_PLAN)
print("계획을 세우는 분석 에이전트 준비 완료")

계획을 세우는 분석 에이전트 준비 완료


이제 같은 복합 요청을 **계획을 세우는 에이전트**에 던지고, 메시지 기록과 **할 일 목록**을 관찰합니다.

In [ ]:
# 같은 모양의 복합 요청을 계획 에이전트(analyst)에 던진다.
# 볼 것은 하나다. 계획을 먼저 세우는가, 그리고 네 가지를 다 해내는가.
plan_q = ("chinook DB 에서 아티스트별 앨범 수와 앨범별 곡 수를 구하고, "
          "앨범이 가장 많은 아티스트 3팀이 전체 앨범에서 차지하는 비중(%)도 계산한 다음, "
          "아티스트별 앨범 수 상위 10팀 막대그래프를 아티스트별_앨범수.jpg 라는 이름으로 저장해줘."
          "먼저 어떻게 처리할지 계획 세우고 진행해")

result = await analyst.ainvoke({"messages": plan_q})
print_trajectory(result)

print("\n---- 세운 계획과 진행 상태 ----")
print("계획(todos)이 있나?:", "todos" in result)
# 계획을 세우지 않은 실행도 있을 수 있으므로 get 으로 안전하게 꺼낸다
for item in result.get("todos", []):
    print(f"- [{item['status']}] {item['content']}")

> 방금 출력된 메시지 기록에서 **`write_todos`** 를 찾아보세요. 대개 **맨 처음** 이 도구로 **할 일 목록**을 세우고,
> 각 단계를 처리한 뒤 다시 `write_todos` 로 상태를 갱신합니다.
> 계획을 **눈에 보이게** 만들면, 복합 요청도 단계를 빠뜨리지 않고 처리할 수 있습니다.

> 계획을 **몇 개로 쪼갤지, 어떤 문장으로 적을지는 모델이 정합니다**. 실제 모델을 부르므로 실행할 때마다 항목 수와 문구가 달라질 수 있습니다.
> 우리가 확인할 것은 "계획이 세워졌고 단계별로 처리됐는가"이지, 항목이 정확히 몇 개인지가 아닙니다.

### 🖐️ 함께 따라하기: 다른 주제로 계획을 세우게 하고 완료 개수 확인

데모는 **앨범·아티스트**를 물었습니다. 이번엔 같은 에이전트에 **장르** 쪽 복합 요청을 던져,
데이터가 바뀌어도 **계획을 세우는 동작은 그대로**임을 확인합니다.

1. `analyst` 에 다음을 던져 결과를 `track_result` 에 담으세요.
   "장르별 곡 수와 장르별 평균 재생시간(분)을 구하고, 곡 수 상위 5개 장르의 막대그래프를 저장해줘."
2. `print_trajectory(track_result)` 로 기록을 찍으세요.
3. `track_result.get('todos', [])` 에서 `status` 가 `'completed'` 인 항목 수를 세어, **완료 개수와 전체 개수**를 출력하세요.

**확인 기준**: `todos` 가 **비어 있지 않고**(계획이 세워졌다는 뜻) 완료 개수가 전체 개수와 같습니다.
항목이 몇 개인지·문구가 무엇인지는 **모델이 매번 새로 정하므로 달라도 정상**입니다.
곡 정보는 `tracks`, 장르는 `genres` 표에 있습니다. 열 이름을 모르면 에이전트가 스스로 `db_describe_table` 로 확인합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) analyst 에 장르별 곡 수·평균 재생시간과 상위 5개 장르 막대그래프를 요청해 track_result 에 담는다
# 2) print_trajectory 로 기록을 찍는다
# 3) track_result 의 todos 에서 status 가 'completed' 인 항목 수를 센다
# 4) 완료 수와 전체 수를 출력한다

---
# 4. 자동 분석 파이프라인: 조회부터 리포트까지

이제 실제 업무 흐름을 흉내 냅니다: **1) 수집 → 2) 분석·통계 → 3) 시각화 → 4) 리포트**.
앞 단원과 다른 점은 **네 단계를 전부 에이전트가 도구로** 처리한다는 것입니다. 리포트 저장까지 도구가 합니다.

> **수집 단계에 대하여**: 앞 시간에는 CSV 를 읽는 파일시스템 MCP 도구(`read_text_file`)가 그 자리에 있었습니다.
> 이번에는 **`db_read_query`** 가 그 일을 합니다. 실무에서는 이 자리에 **사내 DB 조회**나 공공데이터 API 호출이 들어갑니다.
> 파일이 DB 로 바뀔 뿐, 흐름과 "수집 결과가 다음 단계의 근거가 된다"는 성질은 그대로입니다.

<img src="../images/analysis_pipeline_mcp.png" width="820">

*아래 셀을 실행한 뒤, 메시지 기록에 찍힌 도구 이름을 네 칸과 하나씩 맞춰 보세요.
분석·통계와 시각화가 **같은 도구**(`code_run-code`)인 것도 확인하세요. 도구는 하나인데 보내는 코드가 다를 뿐입니다.*

---
## 4. 프롬프트 한 문단이 도구 선택을 바꾼다

같은 질문·같은 도구·같은 모델로 **두 번** 돌립니다. 다른 것은 **시스템 프롬프트뿐**입니다.

| 버전 | 저장에 대한 지시 |
|---|---|
| **A** | "파일로 저장할 때는 `files_write_file` 을 쓴다" 한 문장 |
| **B** | 그림은 `savefig`, 마크다운·텍스트만 `files_write_file` 로 **갈라서** |

볼 것은 하나입니다. **도구 호출 기록에서 그래프를 어느 도구로 저장했는가.**

In [ ]:
PNG_MAGIC = b"\x89PNG"                     # 진짜 PNG 파일은 이 네 바이트로 시작한다
CHART_A = OUTPUT_DIR / "비교_A.png"
CHART_B = OUTPUT_DIR / "비교_B.png"

compare_q = ("invoices 표에서 연도별 매출 합계를 구하고, "
             "연도별 매출 막대그래프를 '{name}' 이라는 이름으로 저장해 줘.")

# 두 버전이 공유하는 앞부분
COMMON = ("너는 데이터 분석 비서다. 표의 조회·집계는 db_read_query 로 SQL 을 실행해 구한다. "
          "숫자를 암산하거나 지어내지 않는다. "
          "코드의 결과에 대한 마지막 줄은 반드시 print 로 출력한다. ")

# A: 저장 방법을 뭉뚱그렸다. 그림에도 files_write_file 을 쓰라는 말로 읽힌다.
prompt_a = COMMON + "파일로 저장할 때는 files_write_file 을 쓴다. 경로는 파일 이름만 적는다."
# B: 그림과 글을 갈라 적었다. 위에서 만든 SYSTEM_BASE 가 그 방식이다.
prompt_b = SYSTEM_BASE

for label, prompt, chart in [("A. 뭉뚱그린 프롬프트", prompt_a, CHART_A),
                             ("B. 갈라 적은 프롬프트", prompt_b, CHART_B)]:
    print(f"\n{'-' * 20} {label} {'-' * 20}")
    chart.unlink(missing_ok=True)          # 지난 실행의 파일을 지우고 시작한다
    agent = create_agent(model, TOOLS, system_prompt=prompt,
                         middleware=[ModelCallLimitMiddleware(run_limit=12, exit_behavior="end")])
    print_trajectory(await agent.ainvoke({"messages": compare_q.format(name=chart.name)}))

    # 도구의 말이 아니라 파일로 확인한다. 0 바이트 파일도 exists() 는 True 다.
    size = chart.stat().st_size if chart.exists() else 0
    is_png = chart.exists() and chart.read_bytes()[:4] == PNG_MAGIC
    print(f"\n[산출물] {chart.name} · 크기 {size} 바이트 · PNG 서명 {is_png}")

> 두 기록에서 그래프를 저장한 도구를 찾아 견주세요.
>
> - **A** 는 `files_write_file` 로 `.png` 를 쓰려다 **빈 파일**을 남기기 쉽습니다.
>   그 도구는 글자만 쓰는데도 서버는 `Successfully wrote to ...` 라고 답합니다.
> - **B** 는 `code_run-code` 안에서 `savefig` 로 저장하므로 파일이 실제로 남습니다.

**교훈**: 도구가 여럿일 때는 **"무엇을 어느 도구로"** 를 갈라 적습니다.
"파일로 저장해라" 처럼 뭉뚱그리면 모델이 엉뚱한 도구를 고릅니다.

**그리고 도구가 성공이라고 답해도 결과물이 멀쩡하다는 뜻은 아닙니다.** 크기와 서명으로 반증해야 압니다.

In [ ]:
# 지난 실행의 산출물을 먼저 지운다. 남아 있으면 이번에 아무것도 안 만들어도 '있음=True' 가 나온다
for name in [CHART_NAME, REPORT_NAME]:
    (OUTPUT_DIR / name).unlink(missing_ok=True)

# 네 단계를 순서까지 못박아 적는다 - 요청이 구체적일수록 계획이 그대로 따라온다
pipeline_q = (
    "1) invoices 표에서 연도별 매출 합계를 조회하고, "
    "2) 전년 대비 증감률을 계산하고, "
    f"3) 연도별 매출 막대그래프를 '{CHART_NAME}' 이라는 이름으로 저장하고, "
    f"4) 위 결과를 정리한 마크다운 리포트를 '{REPORT_NAME}' 이라는 이름으로 저장해줘. "
    "그래프는 matplotlib 으로 그려서 저장해."
)

report_result = await analyst.ainvoke({"messages": pipeline_q})
print_trajectory(report_result)

In [ ]:
# 모델의 '저장했습니다' 라는 말이 아니라 파일로 확인한다
for name in [CHART_NAME, REPORT_NAME]:
    path = OUTPUT_DIR / name
    print(f"{name:28} 있음={path.exists()}")

report_file = OUTPUT_DIR / REPORT_NAME
if report_file.exists():
    print("-" * 40)
    print(report_file.read_text(encoding="utf-8")[:400])

위 메시지 기록에서 **조회(`db_read_query`) → 계산(`code_run-code`) → 그래프(`code_run-code`) → 저장(`files_write_file`)** 이
차례로 나타나는지 확인해 보세요. 이 네 단계가 오늘 자동화하려던 **수집 → 분석·통계 → 시각화 → 리포트** 흐름 그 자체입니다.

> 어떤 도구를 몇 번 부를지는 모델이 매번 새로 판단하므로 기록이 실행마다 조금씩 다릅니다.
> 네 단계 중 빠진 것이 보이면, 질문 문장에서 그 단계를 더 분명히 적어 다시 실행해 보세요. **요청이 구체적일수록 계획이 정확해집니다.**

> 그래프 저장이 실패했다면 대개 두 가지입니다. `matplotlib.use("Agg")` 를 빠뜨렸거나, 한글 폰트를 지정하지 않아 제목이 네모로 나오는 경우입니다.
> 그래서 질문에 그 두 가지를 **미리 적어 두었습니다**. 남의 프로세스에서 도는 코드라 우리가 대신 고쳐 줄 수 없기 때문입니다.

### 🖐️ 함께 따라하기: 파이프라인을 다른 주제로 한 번 더

데모는 **연도별 매출**이었습니다. 이번엔 **국가별 매출**로 같은 네 단계를 돌려 봅니다.

1. `analyst` 에 다음 네 단계를 순서대로 못박아 요청하세요.
   국가별 매출 상위 10개 조회 → 상위 3개국이 전체에서 차지하는 비중(%) 계산 →
   상위 10개국 막대그래프를 `국가별_매출.png` 로, 리포트를 `국가별_리포트.md` 로 저장.
   저장 폴더는 시스템 프롬프트에 이미 적어 두었으니 **파일 이름만** 적으면 됩니다.
   그래프에는 `Agg` 와 한글 폰트 지정을 함께 적어 주세요.
2. `print_trajectory()` 로 기록을 찍고, `todos` 항목 수와 완료 수를 출력하세요.
3. 두 파일이 **실제로 생겼는지** 파이썬으로 확인하세요.

**확인 기준**: 파일 두 개가 모두 생기고, `todos` 의 모든 항목이 `completed` 입니다.
비중(%) 값은 리포트 본문에 숫자로 들어가 있어야 합니다. 모델이 암산했는지 코드로 계산했는지 **기록의 `code_run-code`** 로 확인하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 국가별 매출 상위 10개 조회 → 상위 3개국 비중 계산 → 막대그래프 저장 → 리포트 저장을 한 문장으로 요청한다
# 2) print_trajectory 로 기록을 찍고 todos 의 전체·완료 수를 출력한다
# 3) 두 파일이 실제로 생겼는지 파이썬으로 확인한다

---
# 5. 분석 결과를 정해진 틀로: 표·JSON 에 담기

지금까지 에이전트의 답은 **자유 문장**이었습니다. 사람이 읽기엔 좋지만, 결과를 **표(DataFrame)나 데이터베이스·JSON 에 적재**하려면
매번 문장에서 값을 뽑아내야 합니다. 번거롭고 잘 깨집니다.

19일차에서 배운 **구조화된 출력**을 여기에 적용합니다. 방법은 **두 단계**입니다.

1. 에이전트가 평소처럼 도구로 분석해 **문장으로 답한다**.
2. 그 답을 **`model.with_structured_output(AnalysisResult)`** 로 **정해진 틀로 바꾼다**.

> 정형화(2단계)는 **도구 없이 한 번만** 모델을 부르므로 안전하고 결과가 일정합니다.

In [ ]:
# [제공 코드] 정형화 도구 - 에이전트의 자유 문장 답변을 정해진 틀(AnalysisResult)로 바꾼다
from pydantic import BaseModel, Field


class AnalysisResult(BaseModel):
    """분석 답변을 담는 정해진 틀 - 지표 이름·핵심 수치·한 문장 해석."""
    # 필드 설명은 '이 칸에 무엇이 들어가는가' 를 짧은 명사구로 적는다.
    # "~하라", "~쓰지 마라" 처럼 긴 지시문으로 쓰면 모델이 그 문장을 값으로 베껴 넣는다.
    metric: str = Field(description="집계 대상과 방법이 드러나는 한국어 지표 이름. 예: 매출 1위 국가의 매출 합계")
    value: float = Field(description="핵심 수치 하나. 단위 없는 숫자")
    interpretation: str = Field(description="그 수치를 설명하는 한국어 한 문장")


# with_structured_output: 모델이 답을 AnalysisResult 구조로만 내놓게 한다(도구 없이 단발 호출 -> 안전).
structurer = model.with_structured_output(AnalysisResult)
print("정형화 준비 완료 - 문장을 AnalysisResult 로 바꿉니다")

**1단계: 에이전트가 문장으로 답합니다.**

In [ ]:
# 정형화는 '수치 하나'로 답이 나오는 질문일 때 잘 맞는다
single_q = "국가별 매출 합계에서 가장 매출이 큰 국가와 그 합계를 알려줘."
res_single = await analyst.ainvoke({"messages": single_q})
answer_text = res_single["messages"][-1].text
print("[에이전트 답변]", answer_text)

**2단계: 그 답변을 `AnalysisResult` 틀로 정형화합니다.**

In [ ]:
# 질문과 답변을 함께 넣는다. 답변만 주면 지표 이름이 '매출' 처럼 뭉뚱그려진다.
# 무엇을 집계한 값인지는 질문에 들어 있다. 답변을 함께 주는 것도 중요하다.
# 그래야 이 호출이 새로 답하지 않고 옮겨 담기만 한다(2단계의 핵심).
info = structurer.invoke(f"질문: {single_q}\n답변: {answer_text}")
print("타입:", type(info).__name__)
print("지표(metric):", info.metric)
print("수치(value):", info.value)
print("해석(interpretation):", info.interpretation)

> 자유 문장이 **`AnalysisResult` 객체**로 정리됐습니다. `info.value` 는 진짜 **숫자(float)** 라 바로 계산·비교·저장에 쓸 수 있습니다.
> 문장에서 값을 뜯어낼 필요가 없습니다.

정형화의 목적은 **적재**입니다. 딕셔너리 몇 개를 모으면 그대로 **표(DataFrame)** 가 되고, 표는 **JSON 파일**로 남길 수 있습니다.
사람이 읽는 마크다운 리포트(4절)와 달리 **기계가 읽는** 리포트입니다.

In [ ]:
import pandas as pd

# 딕셔너리 리스트를 그대로 넣으면 키가 열 이름이 된다
metrics_df = pd.DataFrame([info.model_dump()])
display(metrics_df)

# 표를 JSON 으로 저장 - orient='records' 는 '행 하나 = 객체 하나' 형태,
# force_ascii=False 는 한글을 그대로(\uXXXX 로 바꾸지 않고) 쓰기 위한 옵션
metrics_path = OUTPUT_DIR / "metrics.json"
metrics_df.to_json(metrics_path, orient="records", force_ascii=False)
print("지표 저장:", metrics_path)
print(metrics_path.read_text(encoding="utf-8")[:200])

### 🖐️ 함께 따라하기: 다른 지표를 정형화해 두 줄짜리 표로 적재

데모는 **국가별 매출 1위** 하나를 정형화했습니다. 이번엔 지표를 **하나 더** 만들어 두 줄짜리 표로 쌓아 봅니다.

1. `analyst` 에 "직원별 담당 고객 매출 합계에서 1위 직원과 그 합계를 알려줘." 를 던져 답변 문장을 얻으세요.
2. 그 **문장**을 `structurer` 로 정형화하세요.
3. 데모의 `info` 와 방금 만든 결과를 **함께** `pd.DataFrame([...])` 에 넣어 **두 줄짜리 표**를 만들고 출력하세요.
4. 그 표를 `OUTPUT_DIR / "metrics.json"` 에 같은 옵션으로 저장하세요.

**확인 기준**: 표가 **2행 3열**(`metric`·`value`·`interpretation`)이고, `value` 열의 자료형이 **숫자(float)** 입니다.
문장에서는 바로 계산에 못 쓰던 값이 **틀에 담기는 순간 계산 가능한 숫자가 된다**는 것이 이 연습의 요점입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) analyst 에 직원별 담당 고객 매출 1위를 물어 답변 문장을 얻는다
# 2) 그 문장을 structurer 로 정형화한다
# 3) 데모의 info 와 함께 두 줄짜리 DataFrame 을 만들어 출력한다
# 4) OUTPUT_DIR / "metrics.json" 에 저장한다

---
## 이번 강의 정리

| 개념 | 하는 일 |
|---|---|
| 복합 요청의 한계 | 계획 없이 여러 단계를 처리하면 단계를 놓치기 쉬움 |
| 미들웨어 | 에이전트 루프 주변에 기능을 덧붙이는 확장 장치. 정해진 **훅** 자리에만 건다 |
| `TodoListMiddleware` | 복합 요청에 **할 일 목록(계획)** 을 세우고 상태를 추적. `write_todos` 도구를 함께 들고 온다 |
| **Plan-and-Execute** | 계획을 먼저 세우고(Plan) 단계별로 실행(Execute): 위 미들웨어로 얻는 방식의 이름 |
| 자동 분석 파이프라인 | 수집(`db_read_query`) → 분석·통계(`code_run-code`) → 시각화(`code_run-code`) → 리포트(`files_write_file`) |
| 결과 정형화 | 에이전트 답변을 `with_structured_output(AnalysisResult)` 로 정해진 틀로 바꿔 표·JSON 에 적재 |

- 도구가 **손**이라면, 계획(todos)은 에이전트의 **작업 순서표**입니다.
- `result.get('todos', [])` 로 계획과 진행 상태를 **눈으로 확인**할 수 있습니다.
- **도구가 남의 것이어도 계획을 세우는 자리는 그대로**입니다. 미들웨어는 도구가 아니라 **루프**에 걸리기 때문입니다.
- 결과를 다시 쓰려면 **정형화**(에이전트 답변 → `with_structured_output`)로 틀에 맞춰 적재합니다.

## 다음 시간 예고

계획을 세워 여러 단계를 처리하는 데까지 왔습니다.
다음 단원에서는 에이전트의 **출력 품질을 스스로 끌어올리고**, 에이전트가 무엇을 했는지 **추적·운영**하는 방법으로 한 걸음 더 나아갑니다.

수고하셨습니다!